In [3]:
from joblib import load
import numpy as np
import pandas as pd
import os
import sys
import pandas as pd
import numpy as np

bundle = load("rf_final_2008_2022.pkl")

rf_model = bundle["model"]
scaler = bundle["scaler"]
feature_cols = bundle["feature_cols"]
threshold = bundle["threshold"]

print("Model loaded.")
print("Threshold:", threshold)

Model loaded.
Threshold: 0.162539


In [4]:
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)
from src.data_enrichment import get_features

df_feats_2025, _ = get_features("../data/raw")

# Filtrar SOLO temporada 2025
df_2025 = df_feats_2025[df_feats_2025["season_end_year"] == 2025].copy()

print(df_2025.shape)
df_2025.head()


(3504, 79)


,player_id,minutes_played,goals,assists,yellow_cards,second_yellow_cards,direct_red_cards,penalty_goals,matches_played,clean_sheets,...,won_champions,team_ucl_strength,Titles,win_rate,goals_per_game,num_trophies,ballon_dor_winner,player_name,position,main_position
41,1000674,0.0,0.0,0,0,0,0,0,1,0,...,0.0,0.125000,2.0,0.452962,1.655052,0.0,0,Gianluca Prestianni (1000674),Attack - Right Winger,Attack
42,1000675,0.0,0.0,0,0,0,0,0,1,0,...,0.0,0.000000,0.0,0.000000,0.000000,0.0,0,Alejo Sarco (1000675),Attack - Centre-Forward,Attack
79,100131,0.0,0.0,0,1,0,0,0,14,0,...,0.0,0.004016,0.0,0.375000,0.750000,0.0,0,William Carvalho (100131),Midfield - Defensive Midfield,Midfield
100,1003534,0.0,0.0,0,0,0,0,0,1,0,...,0.0,0.000000,0.0,0.000000,0.000000,0.0,0,Marwin Schmitz (1003534),Midfield - Defensive Midfield,Midfield
103,1003925,0.0,0.0,0,0,0,0,0,1,0,...,0.0,0.003135,0.0,0.500000,1.250000,0.0,0,Henrijs Auseklis (1003925),Goalkeeper,Goalkeeper


In [5]:

X_2025 = df_2025[feature_cols].copy()
X_2025_scaled = scaler.transform(X_2025)


In [6]:

proba_2025 = rf_model.predict_proba(X_2025_scaled)[:, 1]
df_2025["proba_win"] = proba_2025


In [7]:

df_2025["predicted_winner"] = (df_2025["proba_win"] >= threshold).astype(int)


In [11]:

top_10_2025 = df_2025.sort_values("proba_win", ascending=False).head(10)

top_10_2025[[
    "player_name",
    "season_end_year",
    "proba_win"
]]


,player_name,season_end_year,proba_win
43670,Raphinha (411295),2025.0,0.036068
27260,Ousmane Dembélé (288230),2025.0,0.035743
8695,Mohamed Salah (148455),2025.0,0.032797
54409,Mateo Retegui (554903),2025.0,0.026670
6158,Harry Kane (132098),2025.0,0.025845
40168,Robert Lewandowski (38253),2025.0,0.015060
64182,Sergi Cardona (678406),2025.0,0.010696
63775,João Neves (670681),2025.0,0.010230
54292,Santiago Gimenez (552955),2025.0,0.008293
52871,İlkay Gündoğan (53622),2025.0,0.007982


In [12]:

winner_2025 = df_2025.loc[df_2025["proba_win"].idxmax()]
print("🏆 Predicted Ballon d’Or 2025 Winner:")
print(winner_2025["player_name"], "with probability:", winner_2025["proba_win"])


🏆 Predicted Ballon d’Or 2025 Winner:
Raphinha (411295) with probability: 0.036067506938274055
